In [16]:
import asyncio
from autogen_ext.models.openai import OpenAIChatCompletionClient
from dotenv import load_dotenv
import os
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.messages import TextMessage
from autogen_agentchat.conditions import TextMentionTermination, MaxMessageTermination

load_dotenv()
api_key = os.getenv('OPENAI_API_KEY')
model_client = OpenAIChatCompletionClient(model='gpt-4o', api_key=api_key)

In [11]:
summary_generator = AssistantAgent(
    name="Summary_Generator",
    model_client=model_client,
    description='A summary generator',
    system_message="You generate summary in under 50 words for the given query."
)

summary_reviewer = AssistantAgent(
    name="Summary_Reviewer",
    model_client=model_client,
    description='A summary rewviewer',
    system_message="You review the summary given by the summary_generator provide feedback to make it better. If you feel that the summary is fine, please say 'APPROVED'"
)

summary_editor = AssistantAgent(
    name="Summary_Editor",
    model_client=model_client,
    description="A summary editor",
    system_message="You revise the summary based on the feedback of summary_reviewer."
)

In [19]:
my_termination = TextMentionTermination(text='APPROVED')  | MaxMessageTermination(max_messages=7)

team = RoundRobinGroupChat(
    participants=[summary_generator, summary_reviewer, summary_editor],
    max_turns=6,
    termination_condition=my_termination   
)

In [20]:
async def run_team():
    task = TextMessage(content='GenAI?',source='user')
    result = await team.run(task=task)
    print(result)

await run_team()

messages=[TextMessage(id='13693542-cea0-4349-933e-6939402b3622', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 12, 16, 16, 40, 260856, tzinfo=datetime.timezone.utc), content='GenAI?', type='TextMessage'), TextMessage(id='ccead257-14b3-4263-b715-0bb1658866bf', source='Summary_Generator', models_usage=RequestUsage(prompt_tokens=273, completion_tokens=51), metadata={}, created_at=datetime.datetime(2025, 7, 12, 16, 16, 41, 834426, tzinfo=datetime.timezone.utc), content='Generative AI (GenAI) refers to AI systems that create new content, such as text, images, and music, by analyzing and learning from existing data. It is used in various fields, including chatbots, art, and personalized content generation.', type='TextMessage'), TextMessage(id='9852dd38-5eec-4c28-aa29-08c7630b1722', source='Summary_Reviewer', models_usage=RequestUsage(prompt_tokens=303, completion_tokens=3), metadata={}, created_at=datetime.datetime(2025, 7, 12, 16, 16, 42, 423190, tzin

messages=

[TextMessage(id='13693542-cea0-4349-933e-6939402b3622', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 12, 16, 16, 40, 260856, tzinfo=datetime.timezone.utc), content='GenAI?', type='TextMessage'), 

TextMessage(id='ccead257-14b3-4263-b715-0bb1658866bf', source='Summary_Generator', models_usage=RequestUsage(prompt_tokens=273, completion_tokens=51), metadata={}, created_at=datetime.datetime(2025, 7, 12, 16, 16, 41, 834426, tzinfo=datetime.timezone.utc), content='Generative AI (GenAI) refers to AI systems that create new content, such as text, images, and music, by analyzing and learning from existing data. It is used in various fields, including chatbots, art, and personalized content generation.', type='TextMessage'), 

TextMessage(id='9852dd38-5eec-4c28-aa29-08c7630b1722', source='Summary_Reviewer', models_usage=RequestUsage(prompt_tokens=303, completion_tokens=3), metadata={}, created_at=datetime.datetime(2025, 7, 12, 16, 16, 42, 423190, tzinfo=datetime.timezone.utc), content='APPROVED', type='TextMessage')] 

stop_reason="Text 'APPROVED' mentioned"